# The join — does the BiGRU add anything to v22?

**Standalone.** Needs only the competition data plus the two pickles. Attach:
`v22_oof.pkl` (from the v22 OOF patch) and `bigru_oof.pkl` + `bigru_folds.pkl`
(from the export notebook) via *Add Input → Upload* or *Notebook Output*.

## The question, stated precisely

Not "does the BiGRU beat v22" — it doesn't; 6.922 vs ~6.55. And not "does the
BiGRU win on some wells" — it will, on maybe a quarter of them. The question is:

> **Can you identify those wells in advance, without the answer key?**

Your earlier classical routing work already found complementarity that was real
but unexploitable — an oracle could route profitably, no observable criterion
could pick the wells. That's the wall. J1 tests whether it's still standing.

## The noise floor, which decides everything

Across v19/v22/v23/v24 the local and leaderboard orderings came out **perfectly
inverted**:

| version | local | LB |
|---|---|---|
| v23 | 6.538 (best) | 8.934 (worst) |
| v19 | 6.554 | 8.926 |
| v24 | 6.5665 | 8.914 |
| v22 | 6.5767 (worst) | 8.913 (best) |

Four for four. With spreads of 0.04 local and 0.02 LB this is almost certainly
noise rather than genuine anti-correlation — but that *is* the point: local
differences at that scale carry **zero** predictive signal for your leaderboard.
So the ship threshold here is **0.20 ft**, and every verdict below enforces it,
with a bootstrap CI alongside so you can see the uncertainty rather than trust a
round number.

## Cells

| Cell | Question | Stop if |
|---|---|---|
| J1 | Are the errors decorrelated, and is the difference *predictable*? | no exploitable signal → stop, don't run J2/J3 |
| J2 | Does a fixed-weight blend beat v22? | gain < 0.20 or CI crosses zero |
| J3 | Does cross-fitted routing beat the best fixed blend? | same |

J1 is a genuine kill-switch. If it comes back negative, that is a **result**, not
a failure — "seven models, one information ceiling, and the classical structure is
the only thing that clears it" is a real finding for the Working Note.

In [ ]:
# ===== J1: error decorrelation + exploitability =====
import pickle, glob, os
import numpy as np, pandas as pd

def _find(name):
    hits = sorted(glob.glob('/kaggle/input/*/%s' % name)) + \
           sorted(glob.glob('/kaggle/input/*/*/%s' % name)) + \
           sorted(glob.glob(name))
    if not hits:
        raise FileNotFoundError(
            '%s not found. Attach it via Add Input -> Upload (or Notebook Output). '
            'Searched /kaggle/input/*/ and the working dir.' % name)
    return hits[0]

V = pickle.load(open(_find('v22_oof.pkl'), 'rb'))
Bg = pickle.load(open(_find('bigru_oof.pkl'), 'rb'))
print('v22   wells: %d   (partial run: %s)' % (len(V['err']), V.get('partial')))
print('bigru wells: %d' % len(Bg['err']))
if V.get('partial'):
    print('  WARNING: v22_oof.pkl is a mid-run checkpoint, not a completed loop.')

W0 = sorted(set(V['err']) & set(Bg['err']))
assert len(W0) > 200, 'only %d wells in common — check both pickles' % len(W0)

# --- CRITICAL: recompute BOTH errors on the STATION INTERSECTION -------------
# The stored per-well `err` values cover DIFFERENT station sets. v22 spans the
# whole blind zone; the BiGRU only stations below win_max (4096) -- the near part,
# before drift accumulates. Comparing the stored numbers charges v22 for the hard
# far toe and the BiGRU for none of it, which inflates the apparent win rate and
# manufactures complementarity that isn't there.
DATA = None
for c in ['/kaggle/input/competitions/rogii-wellbore-geology-prediction',
          '/kaggle/input/rogii-wellbore-geology-prediction']:
    if os.path.isdir(os.path.join(c, 'train')):
        DATA = c; break
if DATA is None:
    _h = glob.glob('/kaggle/input/*/train/*__horizontal_well.csv')
    if _h:
        DATA = os.path.dirname(os.path.dirname(_h[0]))
assert DATA, 'competition data not attached — add it via Add Input -> Competition'
print('data:', DATA)

W, P_v, P_b, TRUTH, WIDX = [], [], [], [], []
_dv, _db = [], []
for w in W0:
    pv, pb = V['pred'].get(w), Bg['pred'].get(w)
    if pv is None or pb is None:
        continue
    common, iv, ib = np.intersect1d(pv['blind_idx'], pb['blind_idx'],
                                    return_indices=True)
    if len(common) < 1:
        continue
    try:
        h = pd.read_csv(os.path.join(DATA, 'train', '%s__horizontal_well.csv' % w),
                        usecols=['TVT'])
    except Exception:
        continue
    tr = h['TVT'].values.astype(float)[common]
    ok = np.isfinite(tr)
    if ok.sum() < 1:
        continue
    _dv.append(1.0 - len(common) / max(len(pv['blind_idx']), 1))
    _db.append(1.0 - len(common) / max(len(pb['blind_idx']), 1))
    W.append(w)
    P_v.append(pv['tvt_hat'][iv][ok].astype(float))
    P_b.append(pb['tvt_hat'][ib][ok].astype(float))
    TRUTH.append(tr[ok]); WIDX.append(np.full(int(ok.sum()), len(WIDX)))

pv_all = np.concatenate(P_v); pb_all = np.concatenate(P_b)
tr_all = np.concatenate(TRUTH); wid_all = np.concatenate(WIDX); nW = len(P_v)

def _well_mae(pred):
    e = np.abs(pred - tr_all)
    return np.array([e[wid_all == i].mean() for i in range(nW)])

ev = _well_mae(pv_all); eb = _well_mae(pb_all)

print('\naligned wells %d | stations %d' % (nW, len(tr_all)))
print('  v22 stations dropped by intersection:   %.1f%%' % (100 * np.mean(_dv)))
print('  bigru stations dropped by intersection: %.1f%%' % (100 * np.mean(_db)))
print('  (a large v22 drop is EXPECTED — the far toe the BiGRU never saw)')
print('\n%-22s %10s %10s' % ('', 'stored', 'on overlap'))
print('  %-20s %10.3f %10.3f'
      % ('v22 mean err', np.mean([V['err'][w] for w in W]), ev.mean()))
print('  %-20s %10.3f %10.3f'
      % ('bigru mean err', np.mean([Bg['err'][w] for w in W]), eb.mean()))
print('  ALL verdicts below use the OVERLAP column.')

# --- decorrelation --------------------------------------------------------
from scipy.stats import pearsonr, spearmanr
_pr = pearsonr(ev, eb); _sr = spearmanr(ev, eb)
print('\nerror correlation: pearson %.3f (p=%.1e)  spearman %.3f (p=%.1e)'
      % (_pr[0], _pr[1], _sr[0], _sr[1]))
print('  high correlation => both fail on the same hard wells => no ensemble edge')

# --- oracle ceiling -------------------------------------------------------
d = ev - eb                                  # >0 where bigru wins
win = float((d > 0).mean())
oracle = float(np.minimum(ev, eb).mean())
print('\nbigru wins on %.1f%% of wells' % (100 * win))
print('ORACLE (perfect routing) %.3f  vs v22 %.3f  => ceiling gain %.3f'
      % (oracle, ev.mean(), ev.mean() - oracle))
print('  the ceiling is what a cheat could get. Everything below must fit under it.')

# --- exploitability: can observables predict the sign of d? ---------------
sig = {}
for nm, src in [('branch_spread', V.get('spread', {})),
                ('nn_dist',       V.get('nn', {})),
                ('field_conf',    V.get('field_conf', {}))]:
    v = np.array([src.get(w, np.nan) for w in W], dtype=float)
    if np.isfinite(v).mean() > 0.5:
        sig[nm] = v

print('\nrouting-signal exploitability (does the signal rank d = e_v22 - e_bigru?)')
print('%-16s %8s %10s %10s %10s' % ('signal', 'finite%', 'spearman', 'p', 'AUC'))
print('-' * 58)
for nm, v in sig.items():
    ok = np.isfinite(v) & np.isfinite(d)
    if ok.sum() < 50:
        continue
    rho, p = spearmanr(v[ok], d[ok])
    y = (d[ok] > 0).astype(int)
    if 0 < y.sum() < len(y):                 # rank-based AUC
        r = pd.Series(v[ok]).rank().values
        n1, n0 = y.sum(), (1 - y).sum()
        auc = (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    else:
        auc = np.nan
    sig[nm] = v
    print('%-16s %7.1f%% %10.3f %10.1e %10.3f'
          % (nm, 100 * np.isfinite(v).mean(), rho, p, auc))

print('\n' + '=' * 62)
_best_auc = 0.5
for nm, v in sig.items():
    ok = np.isfinite(v) & np.isfinite(d)
    if ok.sum() < 50: continue
    y = (d[ok] > 0).astype(int)
    if 0 < y.sum() < len(y):
        r = pd.Series(v[ok]).rank().values
        n1, n0 = y.sum(), (1 - y).sum()
        a = (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
        _best_auc = max(_best_auc, max(a, 1 - a))
print('VERDICT')
if ev.mean() - oracle < 0.20:
    print('  STOP. Even perfect routing gains only %.3f, under the 0.20 noise floor.'
          % (ev.mean() - oracle))
    print('  There is nothing here to exploit. Do not run J2/J3.')
elif _best_auc < 0.58:
    print('  STOP (probably). Oracle ceiling is %.3f, but the best observable'
          % (ev.mean() - oracle))
    print('  separates winners at AUC %.3f — near chance. This is the same' % _best_auc)
    print('  "real but unexploitable" wall as the classical routing work.')
    print('  J2 (fixed blend) may still help slightly; J3 (routing) will not.')
else:
    print('  PROCEED. Oracle ceiling %.3f and best observable AUC %.3f (> 0.58).'
          % (ev.mean() - oracle, _best_auc))
    print('  There is detectable structure. Run J2 and J3.')
print('=' * 62)


## J2 — fixed-weight blend

Station-level blend `(1−a)·v22 + a·bigru`, swept over `a`. Stations are matched by
**row index intersection** per well, so only stations both models actually scored
are counted — no silent misalignment, no imputing one model's prediction where
the other has none.

Truth is read from the training CSVs, since neither pickle stores it.

A bootstrap CI over wells accompanies the point estimate. The gain must clear
0.20 **and** have a CI that excludes zero. One without the other isn't enough.

In [ ]:
# ===== J2: fixed-weight blend sweep =====
# Arrays were built on the station intersection in J1. Reuse them —
# rebuilding here would risk the two cells disagreeing.
base_v, base_b = ev, eb
print('reusing J1 alignment: %d wells, %d stations' % (nW, len(tr_all)))
print('v22 %.4f | bigru %.4f (on the aligned intersection)'
      % (base_v.mean(), base_b.mean()))

print('\n%6s %10s %10s' % ('a', 'MAE', 'vs v22'))
print('-' * 28)
best_a, best_m = 0.0, base_v.mean()
for a in np.arange(0.0, 1.001, 0.05):
    m = _well_mae((1 - a) * pv_all + a * pb_all).mean()
    if abs(a * 20 - round(a * 20)) < 1e-9 and round(a * 20) % 2 == 0:
        print('%6.2f %10.4f %+10.4f' % (a, m, m - base_v.mean()))
    if m < best_m:
        best_a, best_m = float(a), float(m)

gain = base_v.mean() - best_m
per_well_gain = base_v - _well_mae((1 - best_a) * pv_all + best_a * pb_all)
rng = np.random.default_rng(0)
boot = np.array([per_well_gain[rng.integers(0, nW, nW)].mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])

print('\n' + '=' * 62)
print('best a = %.2f  ->  MAE %.4f   gain %.4f   95%% CI [%.4f, %.4f]'
      % (best_a, best_m, gain, lo, hi))

# J1 and J2 measure different things and can point opposite ways. J1's oracle
# ceiling is a WELL-level routing bound: how much you could gain by picking the
# better model per well. J2's gain includes STATION-level variance averaging,
# which accrues even when the two models are structurally identical, purely from
# cancelling independent per-station noise. That is a real effect and a real
# gain -- it is how ensembling works -- but it is NOT complementarity, and it is
# the component least likely to survive to the leaderboard, since v22's dominant
# error is accumulated drift (structured, correlated along the lateral) rather
# than iid noise. Say so explicitly rather than let the two numbers look
# contradictory.
_ceiling = ev.mean() - oracle
if gain > _ceiling + 0.05:
    print('\nNOTE: gain %.4f EXCEEDS J1\'s well-level oracle ceiling %.4f.'
          % (gain, _ceiling))
    print('  So %.4f of it is station-level variance averaging, not' % (gain - _ceiling))
    print('  complementarity. Averaging noise is a legitimate gain, but it is the')
    print('  part most likely to wash out on the LB, where v22\'s dominant error is')
    print('  accumulated drift (correlated along the lateral, not iid).')
    print('  Weight the CI accordingly and treat this as a weaker candidate than')
    print('  the headline number suggests.')

print('VERDICT')
if gain < 0.20:
    print('  DO NOT SHIP. Gain %.4f is under the 0.20 noise floor established by' % gain)
    print('  your four inverted local-vs-LB submissions. This would be a coin flip.')
elif lo <= 0:
    print('  DO NOT SHIP. Gain clears 0.20 but the CI includes zero — not solid.')
else:
    print('  CANDIDATE. Gain %.4f, CI excludes zero. Worth one slot.' % gain)
print('=' * 62)


## J3 — cross-fitted routing

The only variant that would be a genuinely new finding: instead of one global
blend weight, pick per well using an observable.

**Cross-fitted, which is the whole point.** The routing threshold is fit on four
folds and applied to the fifth, never on the wells it's evaluated on. Fitting and
evaluating on the same wells is how "real but unexploitable" complementarity
disguises itself as a win — you'd be reading the answer key through the threshold.

If J1 said the best observable sits near chance, expect this to land at or below
the fixed blend. That outcome is worth recording rather than discarding: it's
direct evidence the complementarity is undetectable ex ante, which is the
substantive claim for the Working Note.

In [ ]:
# ===== J3: cross-fitted routing =====
wells_aligned = [w for w in W if w in V['pred'] and w in Bg['pred']][:nW]
gv = _well_mae(pv_all); gb = _well_mae(pb_all)

# fold labels: reuse the BiGRU split so routing is fit off the evaluated wells
fmap = Bg.get('folds', {})
fold_of = np.array([fmap.get(w, i % 5) for i, w in enumerate(wells_aligned)])
if not fmap:
    print('NOTE: bigru_oof.pkl carried no fold map — using a modulo split.')
print('routing folds:', np.bincount(fold_of))

results = {}
for nm, v_all in sig.items():
    v = np.array([v_all[W.index(w)] for w in wells_aligned], dtype=float)
    if np.isfinite(v).mean() < 0.5:
        continue
    routed = np.empty(len(wells_aligned)); routed[:] = np.nan
    for f in np.unique(fold_of):
        tr_i = np.where((fold_of != f) & np.isfinite(v))[0]
        te_i = np.where((fold_of == f))[0]
        if len(tr_i) < 30 or len(te_i) < 1:
            continue
        # fit: threshold + side that minimises in-fold blended MAE
        best = (np.inf, None, None, None)
        for q in np.arange(0.1, 0.91, 0.05):
            thr = np.nanquantile(v[tr_i], q)
            for side in (+1, -1):
                for a_hi in (0.0, 0.25, 0.5, 0.75, 1.0):
                    sel = (v[tr_i] > thr) if side > 0 else (v[tr_i] < thr)
                    m = np.where(sel, (1 - a_hi) * gv[tr_i] + a_hi * gb[tr_i], gv[tr_i]).mean()
                    if m < best[0]:
                        best = (m, thr, side, a_hi)
        _, thr, side, a_hi = best
        vt = v[te_i]
        sel = np.where(np.isfinite(vt),
                       (vt > thr) if side > 0 else (vt < thr), False)
        routed[te_i] = np.where(sel, (1 - a_hi) * gv[te_i] + a_hi * gb[te_i], gv[te_i])
    ok = np.isfinite(routed)
    results[nm] = (float(routed[ok].mean()), float(gv[ok].mean()), int(ok.sum()))

print('\n%-16s %10s %10s %10s' % ('routing signal', 'routed', 'v22', 'gain'))
print('-' * 50)
for nm, (r, b, n) in sorted(results.items(), key=lambda kv: kv[1][0]):
    print('%-16s %10.4f %10.4f %+10.4f' % (nm, r, b, b - r))

print('\n' + '=' * 62)
print('VERDICT')
if not results:
    print('  No usable routing signal was present in v22_oof.pkl.')
else:
    nm, (r, b, n) = min(results.items(), key=lambda kv: kv[1][0])
    rg = b - r
    print('  best cross-fitted routing: %s, gain %.4f over v22' % (nm, rg))
    if rg < 0.20:
        print('  DO NOT SHIP. Under the 0.20 noise floor.')
        print('  This reproduces the "real but unexploitable" wall: the BiGRU does')
        print('  win on a subpopulation, and no observable identifies it in advance.')
        print('  Record it — that is a finding, and it closes the neural line honestly.')
    elif rg <= gain:
        print('  Routing does not beat the fixed blend (%.4f). Prefer J2.' % gain)
    else:
        print('  CANDIDATE, and beats the fixed blend. This is the interesting case:')
        print('  the routing variable itself is a result worth writing up.')
print('=' * 62)


## Reading the outcome

**Three ways this lands, all informative:**

1. **J1 stops it.** Errors correlated, oracle ceiling small, observables at chance.
   The neural line closes and you've established that a well-built recurrent
   sequence model reaches the same information ceiling as the DP tracker from
   identical inputs — seven models, one ceiling. Publishable as-is.
2. **J2 wins, J3 doesn't.** A fixed blend helps but nothing predicts *where*.
   Ship the blend only if the gain clears 0.20 with a CI excluding zero.
3. **J3 beats J2.** Route rather than blend, and the routing variable becomes the
   headline finding.

**One asymmetry, and it favours you.** The BiGRU's OOF is a true 5-fold held-out
number; v22's benefits from a field built over all training wells minus self,
which is slightly more favourable to v22. So a blend that wins here should win by
at least that much on the leaderboard, not less.

**The truncation caveat, which shapes how you read a negative.** At
`win_max=4096` the sequences cover only roughly the **first half of each blind
zone** — the easier half, nearer the anchor, before drift has accumulated. So this
join is a test on favourable ground, and that gives it a clean decision rule:

- **Errors decorrelate here** → green light, and the full-length rerun becomes
  worth doing properly.
- **Errors stay tightly correlated even on the easy half** → clean negative, and
  you've saved yourself a 20-hour full-length run to learn the same thing.

A negative here is therefore *stronger* evidence than it looks, not weaker.

**One caveat that cuts the other way.** v4's validation didn't score the far toe —
that's what the A–F edits were meant to fix before the run died on the 12-hour
cap. Both models are being compared on the same partially-contaminated region, so
the *comparison* is fair, but the absolute numbers are optimistic for both. Don't
read 6.92 or 6.55 as leaderboard predictions; only read the *difference*.

**And check your dates.** The Working Note deadline in the rules you pasted was
July 6 — if that's still the operative date it's passed, so weigh how much effort
the write-up angle justifies. The competition deadline itself is the one that
matters for J2/J3.